In [24]:
from langgraph.graph import StateGraph, END , START
from langchain_groq import ChatGroq
from typing import TypedDict, List
from dotenv import load_dotenv

In [25]:
load_dotenv()

True

In [26]:
class LLMState(TypedDict):
    query: str
    answer: str
    message_history: List[str]


In [27]:
model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [28]:
graph= StateGraph(LLMState)

In [29]:
def llm_qa(state: LLMState)->LLMState:
    """
    from the user query and message history calls groq model and generates an textual answer
    Args: 
        LLMState:
            query: str
            message_history: List[str]
    Returns: 
        LLMState:
            query: str
            message_history: List[str]
            answer: str
    """
    # extract question from state
    question= state['query']
    # form a prompt
    prompt= f'Answer the following question in brief {question}'

    # ask question to LLM
    answer = model.invoke(prompt).content
    
    # update answer in state
    state['answer']=answer
    return state

In [ ]:
graph.add_node('llm_qa',llm_qa)
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

In [31]:
app=graph.compile()

In [33]:
app.invoke({'query':'explain me langgraph internal DAG in detail'})

{'query': 'explain me langgraph internal DAG in detail',
 'answer': 'The **internal Directed Acyclic Graph (DAG)** in **LangGraph** is a core mechanism for structuring workflows, enabling stateful, real-time applications. Here\'s a detailed breakdown:\n\n---\n\n### **1. Core Components of the DAG**\n- **Nodes**: Represent individual functions or steps in the workflow. Each node encapsulates logic (e.g., data processing, API calls) and may maintain its own state or share a global state.\n- **Edges**: Define dependencies between nodes. A directed edge from Node A to Node B means A must execute before B. This ensures tasks are processed in a valid order, avoiding cycles.\n\n---\n\n### **2. Execution Flow**\n- **Topological Sorting**: The DAG is executed in topological order, ensuring all dependencies of a node are resolved before execution. For example, if Node B depends on Node A, A runs first.\n- **Asynchronous Handling**: Nodes can process tasks concurrently if they are independent (no